# ⚔️ MACRO-DREADNOUGHT: The Self-Mutating MoE Engine
Traditional backpropagation is passive. MACRO-DREADNOUGHT is a custom Mixture-of-Experts (MoE) architecture that calculates its own mathematical confusion, routes data across three distinct computational lanes, and utilizes an active **DNA Mutation Engine** to violently rewrite its own weights during runtime.

This notebook runs the full architecture against the Stanford Tiny ImageNet dataset.

In [1]:
import os
import urllib.request
import zipfile
import shutil
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import numpy as np
import time

# --- 📁 DATASET SETUP & CLEANUP (KAGGLE SAFE) ---
def setup_tiny_imagenet():
    dataset_dir = "./tiny-imagenet-200"
    
    # Only download and extract if it hasn't been done yet
    if not os.path.exists(dataset_dir):
        print("📥 Downloading Tiny ImageNet (This takes a minute)...")
        urllib.request.urlretrieve("http://cs231n.stanford.edu/tiny-imagenet-200.zip", "tiny.zip")
        
        print("📦 Extracting to Kaggle Working Directory...")
        with zipfile.ZipFile("tiny.zip", 'r') as zip_ref:
            zip_ref.extractall(".")
        os.remove("tiny.zip") # Free up disk space
        
        # Fixing the messy validation folder for PyTorch ImageFolder
        print("🧹 Cleaning up validation directory...")
        val_dir = os.path.join(dataset_dir, 'val')
        val_annotations = os.path.join(val_dir, 'val_annotations.txt')
        
        if os.path.exists(val_annotations):
            val_df = pd.read_csv(val_annotations, sep='\t', header=None, names=['File', 'Class', 'X', 'Y', 'H', 'W'])
            for _, row in val_df.iterrows():
                filename, cls = row['File'], row['Class']
                os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
                src = os.path.join(val_dir, 'images', filename)
                dst = os.path.join(val_dir, cls, filename)
                if os.path.exists(src):
                    shutil.move(src, dst)
                    
            # Delete the old unsorted images folder and the text file
            shutil.rmtree(os.path.join(val_dir, 'images'))
            os.remove(val_annotations)
            
    print("✅ Tiny ImageNet Ready!")
    return dataset_dir

## ⚡ Part 1: The Core Physics (SpLR_V2)
Standard ReLUs are static. `SpLR_V2` calculates the real-time Shannon Entropy of the layer and dynamically widens or chokes the mathematical gradient. It acts as a localized, non-linear feature selector.

In [2]:
# --- ⚡ PART 1: CORE PHYSICS ---
class SpLR_V2(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.a = nn.Parameter(torch.ones(1, channels, 1, 1) * 1.1)
        self.b = nn.Parameter(torch.ones(1, channels, 1, 1) * 0.5)
        self.c = nn.Parameter(torch.ones(1, channels, 1, 1) * 0.1) 

    def forward(self, x, entropy=None):
        safe_b = torch.abs(self.b) + 1e-4 
        if entropy is not None:
            dilation_factor = 1.0 - torch.clamp(entropy, 0, 0.99)
            safe_b = safe_b * dilation_factor.view(-1, 1, 1, 1)
        return self.a * x * torch.exp(-safe_b * torch.pow(x, 2)) + self.c * x

def get_entropy(attn_scores):
    epsilon = 1e-8
    return -torch.sum(attn_scores * torch.log(attn_scores + epsilon), dim=1)

## 🧩 Part 2: The MoE Router (HighwayLayerV3)
The layer splits the tensor across three paths:
* **Head A:** The Primary feature extractor.
* **Head B:** The Shadow Mutator (processes pure mathematical error).
* **Head C:** The Ghost Highway (processes wide-angle macro shapes using dilated convolutions).

In [3]:
# --- 🧩 PART 2: THE MOE HIGHWAY LAYER ---
class HighwayLayerV3(nn.Module):
    def __init__(self, channels, ghost_dilation=1, uniform_base=0.45):
        super().__init__()
        self.uniform_base = uniform_base
        self.head_a = nn.Conv2d(channels, channels, 3, padding=1)
        self.head_b = nn.Conv2d(channels, channels, 3, padding=1)
        self.head_c = nn.Conv2d(channels, channels, 3, padding=ghost_dilation, dilation=ghost_dilation)
        
        self.pulse_a, self.pulse_b, self.pulse_c = SpLR_V2(channels), SpLR_V2(channels), SpLR_V2(channels)
        self.compress = nn.Conv2d(channels, channels // 4, kernel_size=1)
        self.expand = nn.Conv2d(channels // 4, channels, kernel_size=1)
        self.router = nn.Linear(channels, 3) 
        self.temporal_gate = nn.Sequential(nn.Linear(channels, channels), nn.Sigmoid())
        
        self.last_x = None
        self.failed_buffer = None

    def forward(self, x, hidden_state=None, forensic_bus=None):
        self.last_x = x.detach()
        batch, c, h, w = x.shape
        pool = F.adaptive_avg_pool2d(x, (1, 1)).view(batch, -1)
        
        attn_scores = (1.0 - self.uniform_base) * F.softmax(self.router(pool), dim=1) + (self.uniform_base / 3.0)
        entropy = get_entropy(attn_scores)
        norm_ent = (entropy / np.log(3)).view(batch, 1, 1, 1)
        
        raw_c = self.head_c(x)
        if forensic_bus is not None: raw_c = raw_c + forensic_bus

        out_a = self.pulse_a(self.head_a(x), norm_ent)
        out_b = self.pulse_b(x - self.head_a(x), norm_ent)
        out_c = self.pulse_c(raw_c, norm_ent)

        combined = (attn_scores[:, 0].view(batch, 1, 1, 1) * out_a +
                    attn_scores[:, 1].view(batch, 1, 1, 1) * out_b +
                    attn_scores[:, 2].view(batch, 1, 1, 1) * out_c)
        
        output = x + (combined * (1.0 - norm_ent))
        z = self.temporal_gate(F.adaptive_avg_pool2d(output, (1, 1)).view(batch, -1)).view(batch, c, 1, 1)
        
        new_bus = (1.0 - z) * output 
        if hidden_state is None: hidden_state = torch.zeros_like(output)
        new_hidden = (1 - z) * hidden_state + z * output
        
        echo_loss = F.mse_loss(self.expand(self.compress(output)), output.detach())
        return new_hidden, new_bus, {"echo": echo_loss, "lero": entropy.mean(), "mono": attn_scores.max(dim=1)[0].mean(), "raw_entropy": entropy.mean()}

    def update_hit_list(self, indices):
        if len(indices) > 0 and self.last_x is not None:
            fails = self.last_x[indices].detach()
            if self.failed_buffer is None:
                self.failed_buffer = fails
            else:
                self.failed_buffer = torch.cat([self.failed_buffer, fails], dim=0)[-64:]

## 🧪 Part 3: The Orchestrator
The physical chassis. It cascades the image through 10 MoE layers, perfectly threading the Temporal Memory Gate and the Forensic Garbage Bus from Layer $N$ to Layer $N+1$.

In [4]:
# --- 🧪 PART 3: THE MACRO-DREADNOUGHT (TINY IMAGENET SCALE) ---
class MacroDreadnought(nn.Module):
    def __init__(self, num_classes=200, depth=10, channels=256, dropout_rate=0.15, uniform_base=0.45):
        super().__init__()
        
        self.vanguard = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), SpLR_V2(64), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, channels, 3, padding=1), nn.BatchNorm2d(channels), SpLR_V2(channels), nn.MaxPool2d(2, 2)
        )
        
        self.highways = nn.ModuleList([
            HighwayLayerV3(channels, ghost_dilation=(i % 2) + 1, uniform_base=uniform_base)
            for i in range(depth)
        ])
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), 
            nn.Flatten(), 
            nn.Dropout(p=dropout_rate), 
            nn.Linear(channels, num_classes)
        )

    def forward(self, x):
        x = self.vanguard(x)
        bus, h_state = None, None
        metrics = []
        for layer in self.highways:
            x, bus, m = layer(x, hidden_state=h_state, forensic_bus=bus)
            h_state = x
            metrics.append(m)
        return self.classifier(x), metrics

    def update_layer_hitlists(self, indices):
        for layer in self.highways:
            layer.update_hit_list(indices)

## ⚔️ Part 4: The Proving Grounds
The master training loop. Notice the 3-Factor Mutation trigger. Every 5 epochs, if the network is highly monopolized (>0.75) and highly confused (>0.40), it intercepts the PyTorch Autograd engine, synthesizes a targeted mutagen from its failed tensors, and completely rewrites the DNA of Head B.

In [ ]:
# --- ⚔️ PART 4: THE PROVING GROUNDS ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔒 THE PERFECT DNA SEQUENCE
BASE_DROPOUT = 0.15     
BASE_LR = 0.01          
BASE_WEIGHT_DECAY = 1e-05
BASE_EXEC_FREQ = 5
BASE_UNIFORM = 0.45
BASE_DIV_PENALTY = 0.20
BASE_LERO_PENALTY = 0.50

EPOCHS = 50

# 1. Setup the data directory safely
data_dir = setup_tiny_imagenet()

# 2. ImageNet Transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

train_dataset = ImageFolder(os.path.join(data_dir, 'train'), transform=train_transform)
test_dataset = ImageFolder(os.path.join(data_dir, 'val'), transform=test_transform)

# 3. Stream from disk to save RAM
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n--- ⚔️ MACRO-DREADNOUGHT vs. TINY IMAGENET ⚔️ ---")
print(f"Dataset: 200 Classes | 64x64 Resolution")
print(f"Architecture: 10 Layers Deep | 256 Channels Wide")
print(f"DNA: LR={BASE_LR} | WD={BASE_WEIGHT_DECAY} | Drop={BASE_DROPOUT} | Uni={BASE_UNIFORM}\n")

model = MacroDreadnought(num_classes=200, depth=10, channels=256, dropout_rate=BASE_DROPOUT, uniform_base=BASE_UNIFORM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=BASE_LR, weight_decay=BASE_WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

criterion_none = nn.CrossEntropyLoss(reduction='none')
criterion_mean = nn.CrossEntropyLoss()

best_test_acc = 0.0
start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    ep_mono, ep_entropy = 0, 0
    train_correct, train_total = 0, 0 
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs, m_list = model(inputs)
        
        losses = criterion_none(outputs, labels)
        preds = outputs.max(1)[1]
        failed_mask = (preds != labels)
        hard_fails = torch.where(failed_mask & (losses > losses.mean()))[0]
        model.update_layer_hitlists(hard_fails)
        
        train_total += labels.size(0)
        train_correct += preds.eq(labels).sum().item()
        
        total_div = 0
        for lyr in model.highways:
            wa, wb, wc = lyr.head_a.weight.view(-1), lyr.head_b.weight.view(-1), lyr.head_c.weight.view(-1)
            total_div += (F.cosine_similarity(wa, wb, dim=0) + F.cosine_similarity(wa, wc, dim=0) + F.cosine_similarity(wb, wc, dim=0)) / 3.0
        
        echo_all = sum(m["echo"] for m in m_list)
        lero_all = sum(m["lero"] for m in m_list)
        
        loss = criterion_mean(outputs, labels) + (BASE_DIV_PENALTY * (total_div/10)) + (0.1 * echo_all) + (BASE_LERO_PENALTY * lero_all)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        ep_mono += m_list[0]["mono"].item()
        ep_entropy += m_list[0]["raw_entropy"].item()

    current_train_acc = 100. * train_correct / train_total
    scheduler.step()
    
    avg_mono = ep_mono / len(train_loader)
    avg_entropy = ep_entropy / len(train_loader)
    
    if avg_mono > 0.75 and avg_entropy > 0.40 and (epoch + 1) % BASE_EXEC_FREQ == 0:
        print(f"  🚨 [K] 3-Factor Align: Mutating... (Mono: {avg_mono:.2f}, Ent: {avg_entropy:.2f})")
        with torch.no_grad():
            for lyr in model.highways:
                wa, wb = lyr.head_a.weight.view(-1), lyr.head_b.weight.view(-1)
                proj = (torch.dot(wb, wa) / (torch.dot(wa, wa) + 1e-8)) * wa
                if lyr.failed_buffer is not None and lyr.failed_buffer.shape[0] > 0:
                    fail_profile = lyr.failed_buffer.mean(dim=(0, 2, 3)) 
                    fail_profile = fail_profile / (fail_profile.norm() + 1e-8)
                    fail_profile = fail_profile.view(1, -1, 1, 1) 
                    targeted_noise_4d = fail_profile * torch.randn_like(lyr.head_b.weight) * 0.05
                    targeted_noise = targeted_noise_4d.view(-1)
                else:
                    targeted_noise = torch.randn_like(wb) * 0.03

                lyr.head_b.weight.copy_((wb - proj + targeted_noise).view_as(lyr.head_b.weight))
                lyr.failed_buffer = None 
    
    model.eval(); tc, tt = 0, 0
    with torch.no_grad():
        for ti, tl in test_loader:
            to, _ = model(ti.to(DEVICE))
            _, tp = to.max(1); tt += tl.size(0); tc += tp.eq(tl.to(DEVICE)).sum().item()
    
    current_test_acc = 100. * tc / tt
    if current_test_acc > best_test_acc:
        best_test_acc = current_test_acc
        
    print(f"  Epoch [{epoch+1:2d}/{EPOCHS}] | Train Acc: {current_train_acc:5.2f}% | Test Acc: {current_test_acc:5.2f}% | Best Test: {best_test_acc:5.2f}%")

elapsed = time.time() - start_time
print(f"\n🏆 MACRO-DREADNOUGHT FINAL PEAK ACCURACY (TINY IMAGENET): {best_test_acc:5.2f}%")
print(f"⏱️ Total Training Time: {elapsed/60:.2f} minutes")

📥 Downloading Tiny ImageNet (This takes a minute)...
📦 Extracting to Kaggle Working Directory...
🧹 Cleaning up validation directory...
✅ Tiny ImageNet Ready!

--- ⚔️ MACRO-DREADNOUGHT vs. TINY IMAGENET ⚔️ ---
Dataset: 200 Classes | 64x64 Resolution
Architecture: 10 Layers Deep | 256 Channels Wide
DNA: LR=0.01 | WD=1e-05 | Drop=0.15 | Uni=0.45

  Epoch [ 1/50] | Train Acc:  7.15% | Test Acc: 12.53% | Best Test: 12.53%
  Epoch [ 2/50] | Train Acc: 14.09% | Test Acc: 16.56% | Best Test: 16.56%
  Epoch [ 3/50] | Train Acc: 16.70% | Test Acc: 17.66% | Best Test: 17.66%
  Epoch [ 4/50] | Train Acc: 18.44% | Test Acc: 17.71% | Best Test: 17.71%
